# 一、基础设置

## （一）导入库

In [1]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from tqdm import tqdm
import pickle

## （二）导入函数

In [18]:
import os
os.chdir('/mnt/workspace/FRD-CLIP')

from simple_tokenizer import SimpleTokenizer
from multimodal_model_new import Ablation1

## （三）基础配置

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
# device="cpu"
BATCH_SIZE = 16
EPOCHS = 30
# LR = 1e-4

# 二、数据准备

## （一）导入地址和函数

In [4]:
# 加载clip
import clip
from PIL import Image

clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()  # CLIP 一般冻结

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [5]:
from transformers import BertTokenizer  # 不是 BertModel

# 1. 加载分词器
bert_tokenizer = BertTokenizer.from_pretrained("/mnt/workspace/FRD-CLIP/models/bert-base-chinese")

class ReviewDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text_clean"]

        # 2. 用分词器编码文本
        bert_enc = bert_tokenizer(
            text,
            max_length=77,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        bert_input_ids = bert_enc["input_ids"].squeeze(0)        # [77]
        bert_attention_mask = bert_enc["attention_mask"].squeeze(0)

        # CLIP 部分保持不变
        clip_input_ids = clip.tokenize([text], truncate=True).squeeze(0)

        image = Image.open(row["img_local_path"]).convert("RGB")
        image_tensor = self.preprocess(image)

        return {
            "bert_input_ids": bert_input_ids,
            "bert_attention_mask": bert_attention_mask,
            "clip_input_ids": clip_input_ids,
            "image_tensor": image_tensor,
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

## （二）加载数据

In [6]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, clip_processor=None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        # 如果你没传入clip_processor，可以在这里定义或者直接用clip包
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text_clean'])
        label = row['label']
        img_path = row['img_local_path']

        # --- 增加容错逻辑 ---
        import os
        if not os.path.exists(img_path):
            # 如果路径不存在，打印警告并返回一个全黑的图片占位符
            # print(f"Warning: File not found {img_path}") # 嫌吵可以关掉
            image_tensor = torch.zeros(3, 224, 224) 
        else:
            try:
                image = Image.open(img_path).convert('RGB')
                image_tensor = self.preprocess(image)
            except Exception as e:
                # 即使文件存在，万一文件损坏也返回空图片
                image_tensor = torch.zeros(3, 224, 224)

        # 剩下的逻辑不变
        bert_tokens = self.tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        clip_tokens = clip.tokenize(text, truncate=True)

        return {
            "bert_input_ids": bert_tokens["input_ids"].flatten(),
            "bert_attention_mask": bert_tokens["attention_mask"].flatten(),
            "clip_input_ids": clip_tokens.flatten(), 
            "image_tensor": image_tensor, 
            "label": torch.tensor(label, dtype=torch.long),
            'text': text,
            'img_path': img_path,
            'orig_idx': idx  # 直接带出它在 DataFrame 里的索引
        }

In [7]:
import pandas as pd
from torchvision import models, transforms

train_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv'), bert_tokenizer), batch_size=32, shuffle=True)
val_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/val_ready.csv'), bert_tokenizer), batch_size=32)
test_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv'), bert_tokenizer), batch_size=32)

# 三、模型准备

## （一）导入地址

In [12]:
from transformers import BertModel, BertTokenizer

# local_model_path = "/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/bert-base-chinese/"

# tokenizer = BertTokenizer.from_pretrained(local_model_path)
# model = BertModel.from_pretrained(local_model_path)

bert = BertModel.from_pretrained("/mnt/workspace/FRD-CLIP/models/bert-base-chinese/")

## （二）初始化模型以及LR设置

In [13]:
from multimodal_model_new import ECommerceMultimodalModel
import torch
import torch.nn as nn
from torch.optim import AdamW                 # ← 补上
from torch.optim.lr_scheduler import CosineAnnealingLR   # ← 补上

device = "cuda:0"
LR = 1e-4

model = ECommerceMultimodalModel(bert_model=bert).to(device)

# 优化点：引入 Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 优化点：使用 AdamW 并设置 weight_decay
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.05)

# 优化点：T_max 设置为 总迭代步数 (epochs * len(dataloader))
total_steps = EPOCHS * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## （三）定义训练和评估函数

In [14]:
def train_one_epoch(model, dataloader, optimizer, criterion, scheduler, device):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()

        # 统一字段名
        bert_input_ids   = batch["bert_input_ids"].to(device)
        bert_attn_mask   = batch["bert_attention_mask"].to(device)
        clip_input_ids   = batch["clip_input_ids"].to(device)
        image_tensor     = batch["image_tensor"].to(device)
        labels           = batch["label"].to(device)

        logits = model(bert_input_ids, bert_attn_mask,
                       clip_input_ids, image_tensor)

        loss = criterion(logits, labels)

        # loss.backward()
        # optimizer.step()
        # running_loss += loss.item()

        loss.backward()
        # 梯度裁剪：防止 Transformer 结构梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()  # 优化点：每个 Step 更新学习率，更平滑

        running_loss += loss.item()

    return running_loss / len(dataloader)

In [15]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        bert_input_ids = batch["bert_input_ids"].to(device)
        bert_attn_mask = batch["bert_attention_mask"].to(device)
        clip_input_ids = batch["clip_input_ids"].to(device)
        image_tensor   = batch["image_tensor"].to(device)
        labels = batch["label"].to(device)

        logits = model(bert_input_ids, bert_attn_mask,
                       clip_input_ids, image_tensor)
        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    return total_loss/len(dataloader), acc, all_true, all_pred

# 四、训练模型

## （一）模型训练

In [19]:
import pandas as pd
import os
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
import numpy as np
from tqdm import tqdm  # 导入 tqdm
from tqdm.auto import tqdm  # 自动选择 notebook 或终端版本


def run_ablation_experiment(mode_name, train_loader, val_loader, device, epochs=20):
    print(f"\n{'='*20} 正在开始实验: {mode_name} {'='*20}")
    
    # 1. 初始化模型
    mode_map = {
        'Text-only': 'text_only',
        'Image-only': 'image_only',
        'Full': 'multimodal'
    }
    
    model = Ablation1(
        mode=mode_map.get(mode_name, 'multimodal'),
        bert_name='/mnt/workspace/FRD-CLIP/models/bert-base-chinese', 
        device=device
    ).to(device)
    
    # 统计可训练参数
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    total_params = sum(p.numel() for p in model.parameters())
    trainable_count = sum(p.numel() for p in trainable_params)
    print(f"总参数量: {total_params:,}, 可训练参数量: {trainable_count:,} ({trainable_count/total_params*100:.2f}%)")
    
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1_1': []}
    best_val_acc = 0
    save_path = f"best_model_{mode_name.replace('-', '_')}.pth"

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch [{epoch}/{epochs}] - 学习率: {optimizer.param_groups[0]['lr']:.6f}")
        
        # 训练 - 带 tqdm 进度条
        train_loss, train_acc = train_one_epoch_ablation(
            model, train_loader, optimizer, criterion, scheduler, device, mode_name
        )
        
        # 验证
        val_loss, val_acc, val_true, val_pred = evaluate_ablation(
            model, val_loader, criterion, device, mode_name
        )
        
        # 记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        report_now = classification_report(val_true, val_pred, output_dict=True, zero_division=0)
        history['val_f1_1'].append(report_now.get('1', report_now.get('fake(1)', {})).get('f1-score', 0))
        
        # 打印 epoch 总结
        print(f"📊 Epoch {epoch} 结果 | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1: {history['val_f1_1'][-1]:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"✅ 保存最优模型 (Val Acc: {val_acc:.4f})")

    # 最终评估
    print(f"\n{'='*20} 实验完成: {mode_name} (加载最优模型进行最终评估) {'='*20}")
    model.load_state_dict(torch.load(save_path))
    final_loss, final_acc, final_true, final_pred = evaluate_ablation(
        model, val_loader, criterion, device, mode_name
    )
    
    final_report = classification_report(
        final_true, final_pred, 
        target_names=['real(0)', 'fake(1)'], 
        digits=4
    )
    
    print(f"Mode: {mode_name} | Best Val Accuracy: {best_val_acc:.4f}")
    print(final_report)
    print("-" * 60)

    del model
    torch.cuda.empty_cache()
    return history, best_val_acc


def train_one_epoch_ablation(model, dataloader, optimizer, criterion, scheduler, device, mode):
    """带 tqdm 进度条和实时指标显示的训练函数"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    # 创建 tqdm 进度条
    pbar = tqdm(dataloader, desc=f"Training [{mode}]", 
                bar_format='{l_bar}{bar:20}{r_bar}{bar:-20b}',
                ncols=100)
    
    for batch_idx, batch in enumerate(pbar):
        # 数据准备
        bert_input_ids = batch['bert_input_ids'].to(device)
        bert_attention_mask = batch['bert_attention_mask'].to(device)
        clip_input_ids = batch.get('clip_input_ids')
        if clip_input_ids is not None:
            clip_input_ids = clip_input_ids.to(device)
        image_tensor = batch['image_tensor'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        # 前向传播（根据模式）
        if mode == 'Text-only':
            logits = model(bert_input_ids, bert_attention_mask, None, None)
        elif mode == 'Image-only':
            dummy_ids = torch.zeros(1, 1, dtype=torch.long, device=device).expand(image_tensor.size(0), 1)
            dummy_mask = torch.ones(1, 1, dtype=torch.long, device=device).expand(image_tensor.size(0), 1)
            logits = model(dummy_ids, dummy_mask, None, image_tensor)
        else:
            logits = model(bert_input_ids, bert_attention_mask, clip_input_ids, image_tensor)
        
        loss = criterion(logits, labels)
        loss.backward()
        
        # 梯度裁剪（可选，防止爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # 计算指标
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        # 实时更新进度条显示
        current_loss = total_loss / (batch_idx + 1)
        current_acc = correct / total
        pbar.set_postfix({
            'Loss': f'{current_loss:.4f}',
            'Acc': f'{current_acc:.4f}',
            'LR': f'{optimizer.param_groups[0]["lr"]:.6f}'
        })
    
    scheduler.step()
    
    avg_loss = total_loss / len(dataloader)
    avg_acc = correct / total
    
    return avg_loss, avg_acc


def evaluate_ablation(model, dataloader, criterion, device, mode):
    """带 tqdm 进度条的评估函数"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds, all_labels = [], []
    
    # 创建 tqdm 进度条
    pbar = tqdm(dataloader, desc=f"Evaluating [{mode}]", 
                bar_format='{l_bar}{bar:20}{r_bar}{bar:-20b}',
                ncols=100, colour='green')
    
    with torch.no_grad():
        for batch in pbar:
            bert_input_ids = batch['bert_input_ids'].to(device)
            bert_attention_mask = batch['bert_attention_mask'].to(device)
            clip_input_ids = batch.get('clip_input_ids')
            if clip_input_ids is not None:
                clip_input_ids = clip_input_ids.to(device)
            image_tensor = batch['image_tensor'].to(device)
            labels = batch['label'].to(device)
            
            # 根据模式
            if mode == 'Text-only':
                logits = model(bert_input_ids, bert_attention_mask, None, None)
            elif mode == 'Image-only':
                dummy_ids = torch.zeros(1, 1, dtype=torch.long, device=device).expand(image_tensor.size(0), 1)
                dummy_mask = torch.ones(1, 1, dtype=torch.long, device=device).expand(image_tensor.size(0), 1)
                logits = model(dummy_ids, dummy_mask, None, image_tensor)
            else:
                logits = model(bert_input_ids, bert_attention_mask, clip_input_ids, image_tensor)
            
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # 实时更新进度条
            current_loss = total_loss / len(pbar)
            current_acc = correct / total
            pbar.set_postfix({
                'Loss': f'{current_loss:.4f}',
                'Acc': f'{current_acc:.4f}'
            })
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy, all_labels, all_preds

### text-only

In [13]:
ablation_modes = ['Text-only', 'Image-only']
all_results = {}
summary_table = []

print(f"\n{'#'*50}")
print(f"开始消融实验，共 {len(ablation_modes)} 个实验")
print(f"实验列表: {', '.join(ablation_modes)}")
print(f"{'#'*50}")

for i, mode in enumerate(ablation_modes):
    print(f"\n\n{'#'*30}")
    print(f"### 进度: [{i+1}/{len(ablation_modes)}] 当前实验: {mode}")
    print(f"{'#'*30}")
    
    hist, best_acc = run_ablation_experiment(mode, train_loader, val_loader, device, epochs=10)
    
    all_results[mode] = hist
    summary_table.append({
        'Experiment': mode,
        'Best Val Acc': f"{best_acc:.4f}",
        'Final F1 (Fake)': f"{hist['val_f1_1'][np.argmax(hist['val_acc'])]:.4f}"
    })

# 打印汇总
df_summary = pd.DataFrame(summary_table)
print(f"\n{'='*20} 消融实验简报 {'='*20}")
print(df_summary.to_string(index=False))

# 保存结果到 CSV
df_summary.to_csv('ablation_results.csv', index=False)
print(f"\n结果已保存到 ablation_results.csv")


##################################################
开始消融实验，共 2 个实验
实验列表: Text-only, Image-only
##################################################


##############################
### 进度: [1/2] 当前实验: Text-only
##############################

==================== 正在开始实验: Text-only ====================
总参数量: 102,564,482, 可训练参数量: 296,834 (0.29%)

Epoch [1/10] - 学习率: 0.000100


Training [Text-only]: 100%|████████████████████| 171/171 [00:20<00:00,  8.31it/s, Loss=0.4939, Acc=0
Evaluating [Text-only]: 100%|████████████████████| 37/37 [00:04<00:00,  8.52it/s, Loss=0.4188, Acc=0 


📊 Epoch 1 结果 | Train Loss: 0.4939 | Train Acc: 0.7625 | Val Loss: 0.4188 | Val Acc: 0.8039 | Val F1: 0.8058
✅ 保存最优模型 (Val Acc: 0.8039)

Epoch [2/10] - 学习率: 0.000100


Training [Text-only]: 100%|████████████████████| 171/171 [00:19<00:00,  8.91it/s, Loss=0.3816, Acc=0
Evaluating [Text-only]: 100%|████████████████████| 37/37 [00:04<00:00,  9.08it/s, Loss=0.3352, Acc=0 


📊 Epoch 2 结果 | Train Loss: 0.3816 | Train Acc: 0.8401 | Val Loss: 0.3352 | Val Acc: 0.8716 | Val F1: 0.8482
✅ 保存最优模型 (Val Acc: 0.8716)

Epoch [3/10] - 学习率: 0.000100


...

Training [Image-only]: 100%|████████████████████| 171/171 [00:13<00:00, 12.63it/s, Loss=0.6906, Acc=
Evaluating [Image-only]: 100%|████████████████████| 37/37 [00:02<00:00, 13.33it/s, Loss=0.6905, Acc= 


📊 Epoch 10 结果 | Train Loss: 0.6906 | Train Acc: 0.5362 | Val Loss: 0.6905 | Val Acc: 0.5377 | Val F1: 0.0000

==================== 实验完成: Image-only (加载最优模型进行最终评估) ====================


Evaluating [Image-only]: 100%|████████████████████| 37/37 [00:02<00:00, 13.35it/s, Loss=0.6908, Acc= 

Mode: Image-only | Best Val Accuracy: 0.5377
              precision    recall  f1-score   support

     real(0)     0.5377    1.0000    0.6993       628
     fake(1)     0.0000    0.0000    0.0000       540

    accuracy                         0.5377      1168
   macro avg     0.2688    0.5000    0.3497      1168
weighted avg     0.2891    0.5377    0.3760      1168

------------------------------------------------------------

==================== 消融实验简报 ====================
Experiment Best Val Acc Final F1 (Fake)
 Text-only       0.8930          0.8788
Image-only       0.5377          0.0000

结果已保存到 ablation_results.csv



/opt/conda/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


image-only不要，真假样本在视觉上不可分

### 其他消融实验（w/o A、w/o W、w/o C、w/o Multi）

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50
import clip
from transformers import BertModel
import numpy as np

class Ablation2(nn.Module):
    """
    多模态假新闻检测消融实验模型
    支持四种消融模式：
    - 'full': 完整模型
    - 'wo_A': 移除动态注意力（平均加权）
    - 'wo_W': 移除相似度加权（Fm 直接使用）
    - 'wo_C': 移除 CLIP（仅用 BERT + ResNet）
    - 'wo_Multi': 移除融合路径（仅 Ft + Fi 拼接）
    """
    
    def __init__(
        self,
        bert_model=None,
        bert_name="bert-base-chinese",
        clip_name="ViT-B/32",
        hidden_dim=256,
        num_classes=2,
        device='cpu',
        mode='full'
    ):
        super().__init__()
        self.device = device
        self.mode = mode
        
        print(f"初始化 Ablation2 - Mode: {mode}")
        
        # ===== 1. BERT 文本编码器（所有模式都需要）=====
        if bert_model is not None:
            self.bert = bert_model
        else:
            self.bert = BertModel.from_pretrained(bert_name)
        for p in self.bert.parameters():
            p.requires_grad = False
        bert_dim = self.bert.config.hidden_size  # 768
        
        # ===== 2. ResNet 视觉编码器（所有模式都需要）=====
        # 注意：所有模式都初始化 ResNet，只是是否使用它的特征不同
        self.resnet = resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()
        
        # 只有 wo_C 和 wo_Multi 真正使用 ResNet 特征，其他模式只是占位
        if mode in ['wo_C', 'wo_Multi']:
            # 这两个模式依赖 ResNet，需要解冻训练
            for p in self.resnet.parameters():
                p.requires_grad = True
        else:
            # full, wo_A, wo_W：ResNet 只是代码兼容性，实际不用其输出
            # 或者如果你想用，可以设为 False
            for p in self.resnet.parameters():
                p.requires_grad = False
        resnet_dim = 2048
        
        # ===== 3. CLIP（wo_C 和 wo_Multi 不需要）=====
        use_clip = (mode not in ['wo_C', 'wo_Multi'])
        if use_clip:
            self.clip_model, _ = clip.load(clip_name, device='cpu')
            self.clip_model.eval()
            for p in self.clip_model.parameters():
                p.requires_grad = False
            clip_dim = self.clip_model.text_projection.shape[1]  # 512
        else:
            clip_dim = 0
            
        # ===== 4. 温度系数（需要 CLIP 的模式）=====
        if mode in ['full', 'wo_A', 'wo_W']:
            self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
        # ===== 5. 投影层设计 =====
        
        if mode == 'wo_C':
            # wo_C: 只用 BERT + ResNet，无 CLIP
            self.proj_text = nn.Linear(bert_dim, hidden_dim)
            self.ln_text = nn.LayerNorm(hidden_dim)
            self.proj_img = nn.Linear(resnet_dim, hidden_dim)
            self.ln_img = nn.LayerNorm(hidden_dim)
            self.fusion_fc = nn.Linear(hidden_dim * 2, hidden_dim)
            
        elif mode == 'wo_Multi':
            # wo_Multi: 无融合路径，仅 Ft + Fi 拼接（都用 ResNet，无 CLIP）
            self.proj_text = nn.Linear(bert_dim, hidden_dim)
            self.ln_text = nn.LayerNorm(hidden_dim)
            self.proj_img = nn.Linear(resnet_dim, hidden_dim)
            self.ln_img = nn.LayerNorm(hidden_dim)
            self.final_proj = nn.Linear(hidden_dim * 2, hidden_dim)
            
        else:
            # full, wo_A, wo_W: 使用 CLIP + BERT + ResNet
            # 文本路径：BERT + CLIP 文本
            self.proj_text = nn.Linear(bert_dim + clip_dim, hidden_dim)
            self.ln_text = nn.LayerNorm(hidden_dim)
            
            # 图像路径：ResNet + CLIP 视觉（注意：这里用了 ResNet！）
            self.proj_img = nn.Linear(resnet_dim + clip_dim, hidden_dim)
            self.ln_img = nn.LayerNorm(hidden_dim)
            
            # 融合路径：CLIP 文本 + CLIP 视觉
            self.proj_clip = nn.Linear(clip_dim * 2, hidden_dim)
            self.ln_clip = nn.LayerNorm(hidden_dim)
            
            # 动态门控（full, wo_W 需要，wo_A 不需要）
            if mode in ['full', 'wo_W']:
                self.gate_fc = nn.Linear(hidden_dim * 3, 3)
        
        # ===== 6. 分类器 =====
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        # 统计可训练参数
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"可训练参数: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    def forward(self, bert_input_ids, bert_attention_mask, clip_input_ids, image_tensor):
        
        # ==================== wo_C: 仅用 BERT + ResNet ====================
        if self.mode == 'wo_C':
            bert_out = self.bert(input_ids=bert_input_ids, attention_mask=bert_attention_mask)
            bert_cls = bert_out.pooler_output
            Ft = self.ln_text(self.proj_text(bert_cls))
            
            # 使用 ResNet（已解冻）
            resnet_vis = self.resnet(image_tensor)
            Fi = self.ln_img(self.proj_img(resnet_vis))
            
            F_fused = self.fusion_fc(torch.cat([Ft, Fi], dim=1))
            logits = self.classifier(F_fused)
            return logits
        
        # ==================== wo_Multi: 无融合路径，仅单模态拼接 ====================
        elif self.mode == 'wo_Multi':
            bert_out = self.bert(input_ids=bert_input_ids, attention_mask=bert_attention_mask)
            bert_cls = bert_out.pooler_output
            Ft = self.ln_text(self.proj_text(bert_cls))
            
            # 只用 ResNet，不用 CLIP
            with torch.no_grad():
                resnet_vis = self.resnet(image_tensor)
            Fi = self.ln_img(self.proj_img(resnet_vis))
            
            F_combined = self.final_proj(torch.cat([Ft, Fi], dim=1))
            logits = self.classifier(F_combined)
            return logits
        
        # ==================== full, wo_A, wo_W: 使用 CLIP 的多模态 ====================
        else:
            # ---- 1) 文本路径：BERT + CLIP 文本 ----
            bert_out = self.bert(input_ids=bert_input_ids, attention_mask=bert_attention_mask)
            bert_cls = bert_out.pooler_output
            
            with torch.no_grad():
                clip_text = self.clip_model.encode_text(clip_input_ids).float()
            clip_text = F.normalize(clip_text, dim=-1)
            Ft = self.ln_text(self.proj_text(torch.cat([bert_cls, clip_text], dim=1)))
            
            # ---- 2) 图像路径：ResNet + CLIP 视觉 ----
            with torch.no_grad():
                # 现在 self.resnet 一定存在！
                resnet_vis = self.resnet(image_tensor)
                clip_vis = self.clip_model.encode_image(image_tensor).float()
            clip_vis = F.normalize(clip_vis, dim=-1)
            Fi = self.ln_img(self.proj_img(torch.cat([resnet_vis, clip_vis], dim=1)))
            
            # ---- 3) 融合路径 ----
            Fm = self.ln_clip(self.proj_clip(torch.cat([clip_text, clip_vis], dim=1)))
            
            # ---- 4) 相似度加权 ----
            if self.mode in ['full', 'wo_A']:
                scale = self.logit_scale.exp()
                sim = F.cosine_similarity(clip_text, clip_vis, dim=1).unsqueeze(1)
                Fm_weighted = (sim * scale) * Fm
            else:  # wo_W
                Fm_weighted = Fm
            
            # ---- 5) 特征聚合 ----
            if self.mode == 'full':
                gate_input = torch.cat([Ft, Fi, Fm_weighted], dim=1)
                alpha = F.softmax(self.gate_fc(gate_input), dim=1)
                F_final = (alpha[:, 0:1] * Ft +
                           alpha[:, 1:2] * Fi +
                           alpha[:, 2:3] * Fm_weighted)
                           
            elif self.mode == 'wo_A':
                # 固定平均
                F_final = (Ft + Fi + Fm_weighted) / 3.0
                
            else:  # wo_W
                gate_input = torch.cat([Ft, Fi, Fm_weighted], dim=1)
                alpha = F.softmax(self.gate_fc(gate_input), dim=1)
                F_final = (alpha[:, 0:1] * Ft +
                           alpha[:, 1:2] * Fi +
                           alpha[:, 2:3] * Fm_weighted)
            
            logits = self.classifier(F_final)
            return logits

### 运行其他消融实验

In [34]:
import pandas as pd
import os
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
import numpy as np
from tqdm import tqdm
from tqdm.auto import tqdm


def run_ablation_experiment(mode_name, train_loader, val_loader, device, epochs=20):
    """
    运行单个消融实验
    
    Args:
        mode_name: 实验名称，如 'full', 'wo_A', 'wo_W', 'wo_C', 'wo_Multi'
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        device: 计算设备
        epochs: 训练轮数
    """
    print(f"\n{'='*20} 正在开始实验: {mode_name} {'='*20}")
    
    # 1. 初始化模型
    model = Ablation2(
        mode=mode_name,  # 直接传递模式名称
        bert_name='/mnt/workspace/FRD-CLIP/models/bert-base-chinese', 
        device=device
    ).to(device)
    
    # 统计可训练参数
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    total_params = sum(p.numel() for p in model.parameters())
    trainable_count = sum(p.numel() for p in trainable_params)
    print(f"总参数量: {total_params:,}, 可训练参数量: {trainable_count:,} ({trainable_count/total_params*100:.2f}%)")
    
    # 优化器设置（wo_C 模式下 ResNet 需要微调）
    if mode_name == 'wo_C':
        optimizer = torch.optim.AdamW([
            {'params': model.resnet.parameters(), 'lr': 1e-5},  # ResNet 用更小学习率
            {'params': [p for n, p in model.named_parameters() if 'resnet' not in n and p.requires_grad], 'lr': 1e-4}
        ], weight_decay=0.01)
    else:
        optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=0.01)
    
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1_1': []}
    best_val_acc = 0
    save_path = f"best_model_{mode_name.replace('-', '_')}.pth"

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch [{epoch}/{epochs}] - 学习率: {optimizer.param_groups[0]['lr']:.6f}")
        
        # 训练
        train_loss, train_acc = train_one_epoch_ablation(
            model, train_loader, optimizer, criterion, scheduler, device, mode_name
        )
        
        # 验证
        val_loss, val_acc, val_true, val_pred = evaluate_ablation(
            model, val_loader, criterion, device, mode_name
        )
        
        # 记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        report_now = classification_report(val_true, val_pred, output_dict=True, zero_division=0)
        history['val_f1_1'].append(report_now.get('1', report_now.get('fake(1)', {})).get('f1-score', 0))
        
        # 打印 epoch 总结
        print(f"📊 Epoch {epoch} 结果 | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1: {history['val_f1_1'][-1]:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"✅ 保存最优模型 (Val Acc: {val_acc:.4f})")

    # 最终评估
    print(f"\n{'='*20} 实验完成: {mode_name} (加载最优模型进行最终评估) {'='*20}")
    model.load_state_dict(torch.load(save_path))
    final_loss, final_acc, final_true, final_pred = evaluate_ablation(
        model, val_loader, criterion, device, mode_name
    )
    
    final_report = classification_report(
        final_true, final_pred, 
        target_names=['real(0)', 'fake(1)'], 
        digits=4
    )
    
    print(f"Mode: {mode_name} | Best Val Accuracy: {best_val_acc:.4f}")
    print(final_report)
    print("-" * 60)

    del model
    torch.cuda.empty_cache()
    return history, best_val_acc


def train_one_epoch_ablation(model, dataloader, optimizer, criterion, scheduler, device, mode):
    """带 tqdm 进度条和实时指标显示的训练函数"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Training [{mode}]", 
                bar_format='{l_bar}{bar:20}{r_bar}{bar:-20b}',
                ncols=100)
    
    for batch_idx, batch in enumerate(pbar):
        # 数据准备
        bert_input_ids = batch['bert_input_ids'].to(device)
        bert_attention_mask = batch['bert_attention_mask'].to(device)
        clip_input_ids = batch.get('clip_input_ids')
        if clip_input_ids is not None:
            clip_input_ids = clip_input_ids.to(device)
        image_tensor = batch['image_tensor'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        # 前向传播（根据模式准备输入）
        logits = forward_by_mode(model, bert_input_ids, bert_attention_mask, 
                                clip_input_ids, image_tensor, mode, device)
        
        loss = criterion(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # 计算指标
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        # 实时更新进度条
        current_loss = total_loss / (batch_idx + 1)
        current_acc = correct / total
        pbar.set_postfix({
            'Loss': f'{current_loss:.4f}',
            'Acc': f'{current_acc:.4f}',
            'LR': f'{optimizer.param_groups[0]["lr"]:.6f}'
        })
    
    scheduler.step()
    
    avg_loss = total_loss / len(dataloader)
    avg_acc = correct / total
    
    return avg_loss, avg_acc


def evaluate_ablation(model, dataloader, criterion, device, mode):
    """带 tqdm 进度条的评估函数"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds, all_labels = [], []
    
    pbar = tqdm(dataloader, desc=f"Evaluating [{mode}]", 
                bar_format='{l_bar}{bar:20}{r_bar}{bar:-20b}',
                ncols=100, colour='green')
    
    with torch.no_grad():
        for batch in pbar:
            bert_input_ids = batch['bert_input_ids'].to(device)
            bert_attention_mask = batch['bert_attention_mask'].to(device)
            clip_input_ids = batch.get('clip_input_ids')
            if clip_input_ids is not None:
                clip_input_ids = clip_input_ids.to(device)
            image_tensor = batch['image_tensor'].to(device)
            labels = batch['label'].to(device)
            
            # 根据模式
            logits = forward_by_mode(model, bert_input_ids, bert_attention_mask,
                                    clip_input_ids, image_tensor, mode, device)
            
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # 实时更新进度条
            current_loss = total_loss / len(pbar)
            current_acc = correct / total
            pbar.set_postfix({
                'Loss': f'{current_loss:.4f}',
                'Acc': f'{current_acc:.4f}'
            })
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy, all_labels, all_preds


def forward_by_mode(model, bert_input_ids, bert_attention_mask, clip_input_ids, 
                   image_tensor, mode, device):
    """
    根据消融模式准备输入并前向传播
    """
    # 不需要 CLIP 的模式：wo_C, wo_Multi
    no_clip_modes = ['wo_C', 'wo_Multi']
    
    # 不需要图像的模式：无（所有模式都需要图像输入，但处理方式不同）
    
    if mode in no_clip_modes:
        # 这些模式不需要 CLIP 输入，传入 dummy
        dummy_clip_ids = torch.zeros(1, 77, dtype=torch.long, device=device).expand(
            image_tensor.size(0), 77)
        return model(bert_input_ids, bert_attention_mask, dummy_clip_ids, image_tensor)
    else:
        # full, wo_A, wo_W：需要 CLIP
        if clip_input_ids is None:
            raise ValueError(f"Mode {mode} requires CLIP input, but got None")
        return model(bert_input_ids, bert_attention_mask, clip_input_ids, image_tensor)


In [35]:
# ==================== 主程序：运行所有消融实验 ====================

def run_all_ablation_experiments(train_loader, val_loader, device, epochs=20):
    """
    运行完整的消融实验套件
    """
    # 定义实验列表
    ablation_modes = ['wo_A', 'wo_W', 'wo_C', 'wo_Multi']
    
    all_results = {}
    summary_table = []

    print(f"\n{'#'*50}")
    print(f"开始消融实验，共 {len(ablation_modes)} 个实验")
    print(f"实验列表: {', '.join(ablation_modes)}")
    print(f"{'#'*50}")

    for i, mode in enumerate(ablation_modes):
        print(f"\n\n{'#'*30}")
        print(f"### 进度: [{i+1}/{len(ablation_modes)}] 当前实验: {mode}")
        print(f"{'#'*30}")
        
        hist, best_acc = run_ablation_experiment(mode, train_loader, val_loader, device, epochs)
        
        all_results[mode] = hist
        summary_table.append({
            'Experiment': mode,
            'Best Val Acc': f"{best_acc:.4f}",
            'Final F1 (Fake)': f"{hist['val_f1_1'][np.argmax(hist['val_acc'])]:.4f}"
        })

    # 打印汇总表格
    df_summary = pd.DataFrame(summary_table)
    print(f"\n{'='*20} 消融实验简报 {'='*20}")
    print(df_summary.to_string(index=False))

    
    return all_results, df_summary


# ==================== 使用示例 ====================

if __name__ == "__main__":
    # 假设你已经有了 train_loader 和 val_loader
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    
    # 运行所有实验
    results, summary = run_all_ablation_experiments(
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=20
    )
    
    # 或者单独运行某个实验
    # history, best_acc = run_ablation_experiment('wo_A', train_loader, val_loader, device, epochs=20)


##################################################
开始消融实验，共 4 个实验
实验列表: wo_A, wo_W, wo_C, wo_Multi
##################################################


##############################
### 进度: [1/4] 当前实验: wo_A
##############################

==================== 正在开始实验: wo_A ====================
初始化 Ablation2 - Mode: wo_A


/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


可训练参数: 1,346,947 / 278,399,940 (0.48%)
总参数量: 278,399,940, 可训练参数量: 1,346,947 (0.48%)

Epoch [1/20] - 学习率: 0.000100


Training [wo_A]: 100%|████████████████████| 171/171 [00:28<00:00,  6.04it/s, Loss=0.5458, Acc=0.7038
Evaluating [wo_A]: 100%|████████████████████| 37/37 [00:05<00:00,  6.19it/s, Loss=0.3576, Acc=0.8459m


📊 Epoch 1 结果 | Train Loss: 0.5458 | Train Acc: 0.7038 | Val Loss: 0.3576 | Val Acc: 0.8459 | Val F1: 0.8182
✅ 保存最优模型 (Val Acc: 0.8459)

Epoch [2/20] - 学习率: 0.000100


...

📊 Epoch 19 结果 | Train Loss: 0.2853 | Train Acc: 0.8821 | Val Loss: 0.2551 | Val Acc: 0.8853 | Val F1: 0.8681

Epoch [20/20] - 学习率: 0.000002


Training [wo_Multi]: 100%|████████████████████| 171/171 [00:21<00:00,  8.06it/s, Loss=0.2870, Acc=0.
Evaluating [wo_Multi]: 100%|████████████████████| 37/37 [00:04<00:00,  8.33it/s, Loss=0.2558, Acc=0. 


📊 Epoch 20 结果 | Train Loss: 0.2870 | Train Acc: 0.8810 | Val Loss: 0.2558 | Val Acc: 0.8861 | Val F1: 0.8682

==================== 实验完成: wo_Multi (加载最优模型进行最终评估) ====================


Evaluating [wo_Multi]: 100%|████████████████████| 37/37 [00:04<00:00,  8.39it/s, Loss=0.2611, Acc=0. 

Mode: wo_Multi | Best Val Accuracy: 0.8878
              precision    recall  f1-score   support

     real(0)     0.8515    0.9586    0.9019       628
     fake(1)     0.9436    0.8056    0.8691       540

    accuracy                         0.8878      1168
   macro avg     0.8975    0.8821    0.8855      1168
weighted avg     0.8941    0.8878    0.8867      1168

------------------------------------------------------------

==================== 消融实验简报 ====================
Experiment Best Val Acc Final F1 (Fake)
      wo_A       0.8938          0.8782
      wo_W       0.8955          0.8822
      wo_C       0.8870          0.8708
  wo_Multi       0.8878          0.8691
